# FlowSight AI — Baseline & Research Setup (Colab)

단일 드론 → **비평면 3D 측위 · 지형 위치에너지 압력 · 장애물 식별** 난제 검증 노트북.
- Part A: CPU 재현(H1/H2 합성 검증)  
- Part B–D: Colab **GPU** 모델(RT-DETRv2 검출 · 메트릭 깊이 · 개방어휘 장애물 + VLM)  
- Part E: H3 검출 파인튜닝 스텁(VisDrone) + Drive 체크포인트

> Runtime ▸ Change runtime type ▸ **GPU (L4/A100 권장)**. 방법론·결과: `experiments/hypotheses.md`.


## 0. 설치 & 코드 가져오기


In [ ]:
# Colab엔 torch 사전설치. 나머지만 설치.
!pip -q install "transformers>=4.46" timm accelerate supervision qwen-vl-utils

# flowsight 코드: (a) Drive에 두었거나 (b) git clone. 아래 경로만 맞추세요.
import sys, os
FLOWSIGHT_DIR = '/content/flowsight'   # 예: git clone 위치 또는 Drive 경로
# !git clone <YOUR_REPO_URL> {FLOWSIGHT_DIR}
if FLOWSIGHT_DIR not in sys.path: sys.path.insert(0, FLOWSIGHT_DIR)
print('python', sys.version.split()[0])


In [ ]:
from google.colab import drive          # 체크포인트/데이터 캐시
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/flowsight_ckpt'; os.makedirs(CKPT_DIR, exist_ok=True)
DATA_DIR = '/content/drive/MyDrive/flowsight_data'; os.makedirs(DATA_DIR, exist_ok=True)
print('ckpt ->', CKPT_DIR)


## Part A — CPU 재현: H1(비평면 측위) · H2(지형 압력 전조)
GPU 없이도 도는 합성 검증. 결과 JSON + figure 생성.


In [ ]:
%cd {FLOWSIGHT_DIR}
!PYTHONPATH=. python experiments/run_h1_positioning.py
!PYTHONPATH=. python experiments/run_h2_pressure.py


In [ ]:
import json, glob
from IPython.display import Image, display
print('H1', json.load(open('experiments/results/h1.json'))['methods']['D_metric_depth'])
print('H2', json.load(open('experiments/results/h2.json'))['lead_vs_crush_s'])
for p in ['experiments/figures/h1_positioning.png','experiments/figures/h2_pressure.png']:
    if os.path.exists(p): display(Image(p))


## Part B — 검출 (RT-DETRv2, GPU)
COCO 사전학습 person 검출 → 발끝점. (밀집 부감은 H3 파인튜닝 필요)


In [ ]:
from transformers.image_utils import load_image
from flowsight.perception.detect import HeadPersonDetector
det = HeadPersonDetector(score_thr=0.4)   # PekingU/rtdetr_v2_r50vd
try:
    img = load_image('https://ultralytics.com/images/zidane.jpg')
except Exception:
    from google.colab import files; up = files.upload(); from PIL import Image as I; img = I.open(list(up)[0])
foot = det.foot_points(img)
print('detections(person):', len(foot)); print(foot[:5])


## Part C — 메트릭 깊이 → 3D 측위 (H1의 실모델 경로)
상대 깊이(DA-V2)를 알려진 앵커로 미터 스케일화 → 카메라 역투영. (네이티브 DA-V2-Metric도 가능)


In [ ]:
from flowsight.geometry.metric_depth import MetricDepth
md_model = MetricDepth()                  # depth-anything/Depth-Anything-V2-Small-hf
depth = md_model.predict(img)
print('depth map', depth.shape, float(depth.min()), float(depth.max()))
# 실제: 지면 앵커 픽셀들의 알려진 카메라-Z로 scale_to_metric() 후,
# PinholeCamera(K,R,C)와 MetricDepth.backproject(cam, foot, metric_depth)로 3D 좌표 복원.
print('TODO: 현장 캘리브(K,R,C) + 앵커 거리 입력 시 3D 측위 활성화')


## Part D — 장애물/지오펜스 + VLM 식별 (H4)


In [ ]:
from flowsight.perception.obstacles import OpenVocabObstacles, SAFETY_PROMPTS
ov = OpenVocabObstacles(box_thr=0.3)      # IDEA-Research/grounding-dino-base
boxes, scores, labels = ov.detect(img, prompts=SAFETY_PROMPTS)
print('obstacle dets:', len(boxes), list(zip(labels or [], (scores).round(2))) [:8])
# 애매 객체 식별(선택, VRAM 큼):
# from flowsight.perception.obstacles import VLMIdentifier
# vlm = VLMIdentifier()  # Qwen/Qwen2.5-VL-7B-Instruct
# print(vlm.identify(img.crop((int(boxes[0][0]),int(boxes[0][1]),int(boxes[0][2]),int(boxes[0][3])))))


## Part E — H3 검출 파인튜닝 스텁 (VisDrone, Drive 체크포인트)
드론 도메인 갭 검증. 데이터 로드 → Trainer. (실행 전 데이터/에폭 조정)


In [ ]:
# 데이터: HF Voxel51/VisDrone2019-DET (또는 Kaggle/공식). person 클래스 카운팅 MAE 평가.
from datasets import load_dataset
# ds = load_dataset('Voxel51/VisDrone2019-DET')   # FiftyOne 포맷 → COCO 변환 필요할 수 있음
from transformers import AutoModelForObjectDetection, AutoImageProcessor, TrainingArguments, Trainer
MODEL='PekingU/rtdetr_v2_r50vd'
proc = AutoImageProcessor.from_pretrained(MODEL)
model = AutoModelForObjectDetection.from_pretrained(MODEL, ignore_mismatched_sizes=True)
args = TrainingArguments(output_dir=CKPT_DIR, per_device_train_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=1e-4, num_train_epochs=30,
    bf16=True, save_strategy='epoch', save_total_limit=2, logging_steps=50,
    resume_from_checkpoint=True, report_to='none')
print('TODO: VisDrone -> COCO형 변환 + collate_fn 연결 후 Trainer(model,args,...).train()')
print('baseline(zero-shot) MAE vs fine-tuned MAE 비교 → hypotheses.md H3에 기록')


---
체크포인트는 `CKPT_DIR`(Drive)에 저장 → 세션 끊겨도 `resume_from_checkpoint`로 재개.
결과는 `experiments/hypotheses.md` 반복 로그에 기록하고 가설을 개선하세요.
